# Week 11 Live Coding: Can You Trust the Analyst?

The case told you to ask the analyst for her track record. She sent it. Today we:

1. Build her calibration table. Do her numbers mean what they say?
2. Score her against someone who just guesses the base rate every time.
3. Build the board's decision, and find which inputs change the answer.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd

rec = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk11_forecasts_and_calibration/data/analyst_record.csv')
rec.head()

Each row is one race she forecast, going back to 2018. `forecast` is the chance she gave the candidate. `won` is 1 if that candidate won and 0 if not. 150 races.

## Part 1: Do her numbers mean what they say?

Same check as last week's turnout model. Sort her forecasts into ranges of 20 points, then compare each range's average forecast with the share that happened.

In [ ]:
# pd.cut puts each forecast into one of these five ranges
ranges = [0, 0.2, 0.4, 0.6, 0.8, 1]
rec['range'] = pd.cut(rec['forecast'], ranges)

# For each range: the average forecast and the share that happened.
# observed=True skips empty ranges.
rec.groupby('range', observed=True)[['forecast', 'won']].mean().round(3)

*Check: in the top range she said 0.880 and 0.750 happened.*

The bottom three ranges land close to what she said. The top two do not: she says 0.72 and 0.61 happens; she says 0.88 and 0.75 happens. Her numbers above 60% are too far from 50%. That is **overconfident**.

Each range holds only 20 to 41 races, so no single range proves much. Our race is 60/40, so split her record at 60% and pool each side.

In [ ]:
# True if she gave the candidate more than 60%
rec['above_60'] = rec['forecast'] > 0.60
rec.groupby('above_60')[['forecast', 'won']].mean().round(3)

At 60% and below (89 races) she said 0.329 and 0.326 happened. Above 60% (61 races) she said 0.775 and 0.656 happened.

**This is the finding to take to the board.** Her forecast for our race is 60/40, and both numbers sit in the range where her record is good.

### Does she beat just guessing?

The Brier score is the average of (forecast − outcome)². Lower is better. Compare it with always guessing the base rate.

In [ ]:
# ** 2 squares each gap; .mean() averages over the 150 races
brier_analyst = ((rec['forecast'] - rec['won']) ** 2).mean()

base_rate = rec['won'].mean()   # how often the candidate won
brier_base = ((base_rate - rec['won']) ** 2).mean()

print('base rate:              ', round(base_rate, 3))
print('Brier, her forecasts:   ', round(brier_analyst, 4))
print('Brier, guess base rate: ', round(brier_base, 4))

**0.2117 against 0.2484.** She beats the base rate, so she is not just guessing. But only by about as much as 150 races can show, so do not oversell it.

## Part 2: The decision

The inputs from the case. The last one, what A signs once you have crossed her, is the number we made up.

In [ ]:
# Chance of winning the primary
p_A = 0.60
p_B = 0.40

# Chance of winning November, if nominated
gen_A = 0.45
gen_B = 0.55

# Share of the order each one signs
signs_A = 0.40
signs_B = 1.00

# What A signs if you endorse B (we made this up)
signs_A_crossed = 0.30

# Stay neutral: chance x share for each candidate, added up
neutral = p_A * gen_A * signs_A + p_B * gen_B * signs_B
print('stay neutral (% of the order):', round(neutral * 100, 1))

*Check: 32.8%.*

Now endorse B, but suppose it moves no one (*d* = 0). The only change is that A signs less.

In [ ]:
endorse = p_A * gen_A * signs_A_crossed + p_B * gen_B * signs_B
behind = (neutral - endorse) * 100   # in points

print('endorse B, d = 0 (% of the order):', round(endorse * 100, 1))
print('behind by (points):               ', round(behind, 1))

Each point of primary chance you move from A to B: B gains it, A loses it.

In [ ]:
per_point = gen_B * signs_B - gen_A * signs_A_crossed
print('one moved point is worth:', round(per_point, 3))

# How many points to make up the gap?
break_even = behind / per_point
print('break-even:', round(break_even, 1), 'points of primary chance')

*Check: 30.1%, 2.7 behind, 0.415 a point, break-even 6.5.*

### Change one input

Change one number in the inputs cell, then rerun it and every cell below it (**Runtime → Run cell and below**). Put it back before you try the next one.

1. A doesn't mind being crossed: `signs_A_crossed = 0.40`. *Check: break-even 0.0.*
2. The analyst understates A: `p_A = 0.75` and `p_B = 0.25`. *Check: break-even 8.1.*

The forecast we spent Part 1 testing moves the answer less than 2 points. The number we made up moves it all the way to zero.